# Earnings-Call and Filing NLP Signal: Results

Run `python -m nlp_signal.cli analyze` first (or `demo` for synthetic data). Set `OUT` below to the folder containing `results.json`.

In [ ]:
import json, pandas as pd
from pathlib import Path
from IPython.display import Image, Markdown, display
OUT = Path('../outputs')   # use Path('../outputs/demo') for the synthetic demo
res = json.loads((OUT/'results.json').read_text())
display(Markdown((OUT/'results.md').read_text()))

## 1. Does the signal predict abnormal returns?
Quintile spread, event-time path, information coefficient by quarter.

In [ ]:
for f in ['quintile_car.png','event_path.png','ic_by_period.png']:
    if (OUT/f).exists(): display(Image(str(OUT/f)))

## 2. Which features carry the signal?
Coefficients are per 1 s.d. of each feature's quarter-over-quarter change, SEs clustered by quarter.

In [ ]:
display(Image(str(OUT/'feature_coefs.png')))
h = max(map(int, res['horizons']))
pd.DataFrame(res['horizons'][str(h)]['regression_features']).T.round(4)

## 3. Out-of-sample check
Feature weights are fitted only on filings before the split date. Compare the pre-registered equal-weight signal with the fitted one.

In [ ]:
oos = res['horizons'][str(h)]['oos']
print({k: oos[k] for k in ('split','n_train','n_test')})
pd.DataFrame({k: oos[k] for k in ('signal_prior','signal_fit') if k in oos}).round(4)

## 4. Inspect the data
Read individual documents' scores before trusting any aggregate number.

In [ ]:
sig = pd.read_csv(OUT/'signal_frame.csv', parse_dates=['filing_date'])
ev = pd.read_csv(OUT/'events.csv')
df = sig.merge(ev[['accession','car_20']], on='accession')
df[['ticker','filing_date','source','signal_prior','car_20']].sort_values('signal_prior').head(10)

In [ ]:
df[['ticker','filing_date','source','signal_prior','car_20']].sort_values('signal_prior').tail(10)

## 5. Robustness ideas to try
- Re-run `analyze --offset 1` (skip the reaction day) and compare.
- Run `--sources release` or `--sources transcript` separately.
- Compare `--extractor lexicon` vs `llm`: does the LLM add anything beyond word counts?
- Vary `--cost-bps`; split at different dates.